# Defunciones DANE: totales territoriales de 2008–2025

Consulta de `mortality.parquet`, con los dieciocho años procesados y el esquema común de 30 columnas. Cada fila representa el total de todas las causas para un municipio/país o para un departamento/categoría.

- En 2008–2019 se excluyen `00` y el índice. Se conservan únicamente filas con B=`TOTAL`; `total` procede de C. Los encabezados se localizan en 11–12, 13–14 o 12–13 según el año.
- En 2020–2025 se lee `Cuadro5`, C:AJ, encabezados 8–9. Se excluye el total nacional y se conservan únicamente filas con E=`TOTAL` o `TOTAL MUN`; `total` procede de F. C contiene el departamento y D el municipio. `Total Dpto` o D vacía con departamento presente se normaliza a `Total`.
- Se separan los códigos de los nombres, se conservan ceros iniciales y se verifica el prefijo departamental del municipio. Los totales departamentales usan su código de dos dígitos; extranjeros usan `-1` y sin información `-2`. Las categorías especiales tienen código de departamento nulo.
- Desde 2015 se suman **65–84 + 85–99 + 100 y más**, por sexo, en `edad_65_mas_*`; la edad desconocida se conserva aparte. Una suma con algún componente ausente queda nula.

Para generar el archivo desde la raíz del repositorio:
```bash
.venv/bin/python scripts/processing/process_mortality_dane.py
```

**No sumar las filas `Total` junto con sus desgloses**: se duplicarían las defunciones. El script y este notebook validan únicamente la columna `total` por año y departamento/categoría. La versión local de 2019 es preliminar, con reportes hasta el 31 de julio de 2020.

In [7]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# Funciona desde la raíz del repositorio o desde notebooks/.
root = Path.cwd().resolve()
root = next(
    (path for path in (root, *root.parents)
     if (path / "scripts/processing/process_mortality_dane.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Ejecute el notebook dentro del repositorio data-processing")

parquet_path = root / "data/processed/births_mortality_dane/mortality/mortality.parquet"
if not parquet_path.is_file():
    raise FileNotFoundError(
        f"No existe {parquet_path}. Ejecute scripts/processing/process_mortality_dane.py"
    )

mortality = pd.read_parquet(parquet_path)
print(f"Archivo: {parquet_path}")
print(f"Filas: {len(mortality):,} | Columnas: {len(mortality.columns)}")

Archivo: /home/diego/repos/data-processing/data/processed/births_mortality_dane/mortality/mortality.parquet
Filas: 21,626 | Columnas: 30


## Estructura y schema

In [8]:
print(pq.read_schema(parquet_path))

schema = pd.DataFrame({
    "columna": mortality.columns,
    "tipo_pandas": mortality.dtypes.astype(str).to_numpy(),
    "nulos": mortality.isna().sum().to_numpy(),
})
display(schema)

ano: int64
codigo_departamento: large_string
nombre_departamento: large_string
codigo_municipio: large_string
nombre_municipio: large_string
total: int64
total_hombres: int64
total_mujeres: int64
total_indeterminado: int64
edad_menor_1_hombres: int64
edad_menor_1_mujeres: int64
edad_menor_1_indeterminado: int64
edad_1_4_hombres: int64
edad_1_4_mujeres: int64
edad_1_4_indeterminado: int64
edad_5_14_hombres: int64
edad_5_14_mujeres: int64
edad_5_14_indeterminado: int64
edad_15_44_hombres: int64
edad_15_44_mujeres: int64
edad_15_44_indeterminado: int64
edad_45_64_hombres: int64
edad_45_64_mujeres: int64
edad_45_64_indeterminado: int64
edad_65_mas_hombres: int64
edad_65_mas_mujeres: int64
edad_65_mas_indeterminado: int64
edad_desconocida_hombres: int64
edad_desconocida_mujeres: int64
edad_desconocida_indeterminado: int64
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 4133


,columna,tipo_pandas,nulos
0,ano,Int64,0
1,codigo_departamento,string,720
2,nombre_departamento,string,0
3,codigo_municipio,string,0
4,nombre_municipio,string,0
5,total,Int64,0
6,total_hombres,Int64,0
7,total_mujeres,Int64,0
8,total_indeterminado,Int64,0
9,edad_menor_1_hombres,Int64,0


## Primeras 10 filas con encabezados

In [9]:
with pd.option_context("display.max_columns", None, "display.max_colwidth", 80):
    display(mortality.head(10))

,ano,codigo_departamento,nombre_departamento,codigo_municipio,nombre_municipio,total,total_hombres,total_mujeres,total_indeterminado,edad_menor_1_hombres,edad_menor_1_mujeres,edad_menor_1_indeterminado,edad_1_4_hombres,edad_1_4_mujeres,edad_1_4_indeterminado,edad_5_14_hombres,edad_5_14_mujeres,edad_5_14_indeterminado,edad_15_44_hombres,edad_15_44_mujeres,edad_15_44_indeterminado,edad_45_64_hombres,edad_45_64_mujeres,edad_45_64_indeterminado,edad_65_mas_hombres,edad_65_mas_mujeres,edad_65_mas_indeterminado,edad_desconocida_hombres,edad_desconocida_mujeres,edad_desconocida_indeterminado
0,2008,05,ANTIOQUIA,05,Total,29118,16435,12682,1,672,535,0,155,132,0,201,132,0,4106,1131,0,3565,2351,0,7654,8378,0,82,23,1
1,2008,05,ANTIOQUIA,05001,MEDELLÍN,12329,6596,5732,1,202,179,0,50,40,0,61,40,0,1563,469,0,1537,1037,0,3166,3965,0,17,2,1
2,2008,05,ANTIOQUIA,05002,ABEJORRAL,109,50,59,0,3,2,0,1,1,0,1,1,0,8,5,0,13,13,0,24,37,0,0,0,0
3,2008,05,ANTIOQUIA,05004,ABRIAQUÍ,9,6,3,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,3,3,0,0,0,0
4,2008,05,ANTIOQUIA,05021,ALEJANDRÍA,19,12,7,0,1,0,0,0,0,0,0,1,0,2,1,0,3,0,0,6,5,0,0,0,0
5,2008,05,ANTIOQUIA,05030,AMAGÁ,159,98,61,0,1,0,0,0,0,0,0,0,0,28,6,0,23,16,0,46,39,0,0,0,0
6,2008,05,ANTIOQUIA,05031,AMALFI,120,68,52,0,2,5,0,1,0,0,0,0,0,24,9,0,12,9,0,28,29,0,1,0,0
7,2008,05,ANTIOQUIA,05034,ANDES,221,124,97,0,0,5,0,3,2,0,1,1,0,25,6,0,35,16,0,60,67,0,0,0,0
8,2008,05,ANTIOQUIA,05036,ANGELÓPOLIS,24,15,9,0,0,0,0,0,0,0,0,0,0,2,0,0,2,3,0,11,6,0,0,0,0
9,2008,05,ANTIOQUIA,05038,ANGOSTURA,61,35,26,0,3,2,0,0,1,0,0,0,0,8,2,0,7,4,0,17,17,0,0,0,0


## Validación de total por año y departamento/categoría

Se suman únicamente los valores de `total` de las filas distintas de `Total`, agrupando por año, código y nombre del departamento/categoría. La suma se compara con el total departamental o de categoría guardado en `Total`. Se incluyen las filas municipales sin información y los países extranjeros.

En 2008–2019, `total` corresponde a C del Excel; en 2020–2025, a F. La validación no suma filas de causas ni vuelve a contar el total nacional excluido.

Los grupos con solo la fila `Total` se muestran como **sin desglose** y no se comparan contra cero. La categoría sin información carece de desglose en todos los años. Bogotá también carece de él en 2009–2018; en 2008 y desde 2019 sí incluye el municipio `11001`. Los códigos `-1` y `-2` pueden repetirse y no se usan solos para agrupar.

In [10]:
group_columns = ["ano", "codigo_departamento", "nombre_departamento"]
is_total = mortality["nombre_municipio"].str.casefold().eq("total")
reported = mortality.loc[is_total, group_columns + ["total"]].rename(
    columns={"total": "total_publicado"}
)
assert not reported.duplicated(group_columns).any(), "Hay más de un Total por grupo"

detail = mortality.loc[~is_total]
# dropna=False conserva las categorías sin código de departamento.
sums = detail.groupby(group_columns, dropna=False).agg(
    suma_desglose=("total", lambda values: values.sum(min_count=1)),
    filas_desglose=("total", "size"),
    valores_presentes=("total", "count"),
).reset_index()
validation = reported.merge(sums, on=group_columns, how="outer", indicator=True)
assert not validation["_merge"].eq("right_only").any(), "Hay desgloses sin fila Total"
validation["filas_desglose"] = validation["filas_desglose"].fillna(0).astype(int)
validation["diferencia"] = validation["suma_desglose"] - validation["total_publicado"]
validation["estado"] = "sin desglose"
has_detail = validation["filas_desglose"].gt(0)
comparable = (
    has_detail & validation["valores_presentes"].eq(validation["filas_desglose"])
    & validation["total_publicado"].notna()
)
validation.loc[has_detail | validation["total_publicado"].isna(), "estado"] = "datos incompletos"
validation.loc[comparable, "estado"] = "diferencia"
validation.loc[comparable & validation["diferencia"].eq(0), "estado"] = "coincide"
validation = validation.drop(columns=["_merge", "valores_presentes"])

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(validation)

,ano,codigo_departamento,nombre_departamento,total_publicado,suma_desglose,filas_desglose,diferencia,estado
0,2008,05,ANTIOQUIA,29118,29118,126,0,coincide
1,2008,08,ATLÁNTICO,8958,8958,24,0,coincide
2,2008,11,BOGOTÁ,27698,27698,1,0,coincide
3,2008,13,BOLÍVAR,6086,6086,47,0,coincide
4,2008,15,BOYACÁ,6583,6583,124,0,coincide
5,2008,17,CALDAS,5904,5904,28,0,coincide
6,2008,18,CAQUETÁ,1690,1690,17,0,coincide
7,2008,19,CAUCA,5266,5266,43,0,coincide
8,2008,20,CESAR,3674,3674,26,0,coincide
9,2008,23,CÓRDOBA,4678,4678,31,0,coincide


In [11]:
print("Comparación de total por año y departamento/categoría:")
display(validation.groupby(["ano", "estado"]).size().unstack(fill_value=0))

issues = validation.loc[validation["estado"].isin(["diferencia", "datos incompletos"])]
if not issues.empty:
    display(issues)

# Usa solo las filas Total para evitar duplicar los desgloses.
print("Defunciones publicados por año (suma de las filas Total):")
display(reported.groupby("ano")["total_publicado"].sum().to_frame())
assert issues.empty, "Hay diferencias o datos incompletos: revise la tabla de validación"
print("Todas las comparaciones con desglose coinciden con los totales publicados.")

Comparación de total por año y departamento/categoría:


estado,coincide,sin desglose
ano,,
2008,34,1
2009,33,2
2010,33,2
2011,33,2
2012,33,2
2013,33,2
2014,33,2
2015,33,2
2016,33,2


Defunciones publicados por año (suma de las filas Total):


,total_publicado
ano,
2008,196943
2009,196933
2010,200524
2011,195823
2012,199756
2013,203071
2014,210051
2015,219472
2016,223078


Todas las comparaciones con desglose coinciden con los totales publicados.
